In [3]:
!pip install transformers datasets peft accelerate evaluate scikit-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00


In [4]:
import os
print(os.getcwd())
print(os.listdir())


/content
['.config', 'sample_data']


In [5]:
from google.colab import files
uploaded = files.upload()


Saving not-stereotype.txt to not-stereotype.txt
Saving stereotype.txt to stereotype.txt


In [6]:
with open("stereotype.txt") as f:
    stereotype = f.read().splitlines()


In [7]:
with open("not-stereotype.txt") as f:
    not_stereotype = f.read().splitlines()

print("Stereotype samples:", stereotype[:3])
print("Not stereotype samples:", not_stereotype[:3])
print("Counts ->", len(stereotype), len(not_stereotype))


Stereotype samples: ['In some families , the mother also goes out to work and earn money . ', 'The women do the household chores as well as take part in cultivation . ', 'Men wear trousers and shirts . ']
Not stereotype samples: ['Find out more about the history and various organs of the United Nations. ', 'As in the above activity, count the number of different animals and plants in this area and the number of individuals of each species. ', 'Fig. . (a): A Coniferous Forest ']
Counts -> 200 10000


In [8]:
import pandas as pd

data = pd.DataFrame({
    "text": stereotype + not_stereotype,
    "label": [1]*len(stereotype) + [0]*len(not_stereotype)
})

print(data.head())
print("Total samples:", len(data))


                                                text  label
0  In some families , the mother also goes out to...      1
1  The women do the household chores as well as t...      1
2                    Men wear trousers and shirts .       1
3  For example , if the home manager is intereste...      1
4  The mother should never express her displeasur...      1
Total samples: 10200


In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    stratify=data["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


Train size: 8160
Test size: 2040


In [10]:
!pip install transformers datasets peft accelerate evaluate


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Model loaded successfully")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight        

Model loaded successfully


In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 296,450 || all params: 184,720,132 || trainable%: 0.1605


In [13]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [14]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8160 [00:00<?, ? examples/s]

Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

In [15]:
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # smaller = faster
    per_device_eval_batch_size=4,
    num_train_epochs=1,              # 🔥 reduce from 3 → 1
)

In [18]:
train_dataset = train_dataset.select(range(200))
test_dataset = test_dataset.select(range(50))

In [19]:
print("model exists:", 'model' in globals())
print("train_dataset exists:", 'train_dataset' in globals())
print("training_args exists:", 'training_args' in globals())
print("compute_metrics exists:", 'compute_metrics' in globals())

model exists: True
train_dataset exists: True
training_args exists: True
compute_metrics exists: False


In [20]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [21]:
print("compute_metrics exists:", 'compute_metrics' in globals())

compute_metrics exists: True


In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [23]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=1,   # 🔥 reduce to 1
)

In [25]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    return "Stereotype" if pred == 1 else "Not Stereotype"

print(predict("Women are bad drivers"))
print(predict("Everyone deserves equal rights"))

Not Stereotype
Not Stereotype


In [26]:
model.save_pretrained("stereotype_model")
tokenizer.save_pretrained("stereotype_model")

('stereotype_model/tokenizer_config.json', 'stereotype_model/tokenizer.json')

In [27]:
print(predict("Women are emotional and weak"))
print(predict("Men should not cry"))
print(predict("Everyone deserves equal opportunities"))
print(predict("All people deserve respect"))

Not Stereotype
Not Stereotype
Stereotype
Not Stereotype
